In [1]:
def naive(p, t):
    occurrences = []
    for i in range(len(t) - len(p) + 1):  # loop over alignments
        match = True
        for j in range(len(p)):  # loop over characters
            if t[i+j] != p[j]:  # compare characters
                match = False
                break
        if match:
            occurrences.append(i)  # all chars matched; record
    return occurrences

In [2]:
def reverseComplement(s):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N': 'N'}
    t = ''
    for base in s:
        t = complement[base] + t
    return t

In [3]:
def readGenome(filename):
    genome = ''
    with open(filename, 'r') as f:
        for line in f:
            # ignore header line with genome information
            if not line[0] == '>':
                genome += line.rstrip()
    return genome

In [4]:
def readFastq(filename):
    sequences = []
    qualities = []
    with open(filename) as fh:
        while True:
            fh.readline()  # skip name line
            seq = fh.readline().rstrip()  # read base sequence
            fh.readline()  # skip placeholder line
            qual = fh.readline().rstrip() # base quality line
            if len(seq) == 0:
                break
            sequences.append(seq)
            qualities.append(qual)
    return sequences, qualities

In [6]:
def naive_with_rc(p, t):
    occurrences = naive(p, t)
    if not p == reverseComplement(p):
        occurrences += naive(reverseComplement(p), t)
    return occurrences

In [7]:
t = readGenome('lambda_virus.fa')
occurrences = naive_with_rc('AGTCGA', t)
print('Number of times AGTCGA occurs in lambda virus genome:', len(occurrences))
print('Positions:', occurrences)
print("left most position:", min(occurrences))

Number of times AGTCGA occurs in lambda virus genome: 9
Positions: [18005, 23320, 33657, 44806, 450, 1908, 2472, 41927, 45369]
left most position: 450


In [8]:
#make a new version of the naive called naive_2mm naive_2mm that allows up to 2 mismatches per occurrence
def naive_2mm(p, t):
    occurrences = []
    for i in range(len(t) - len(p) + 1):  # loop over alignments
        match = True
        mismatches = 0
        for j in range(len(p)):  # loop over characters
            if t[i+j] != p[j]:  # compare characters
                mismatches += 1
                if mismatches > 2:
                    match = False
                    break
        if match:
            occurrences.append(i)  # all chars matched; record
    return occurrences

In [9]:
t = readGenome('lambda_virus.fa')
occurrences = naive_2mm('AGGAGGTT', t)
print('Number of times AGGAGGTT occurs in lambda virus genome with up to 2 mismatches:', len(occurrences))
print('Positions:', occurrences)
print("left most position:", min(occurrences))
    


Number of times AGGAGGTT occurs in lambda virus genome with up to 2 mismatches: 215
Positions: [49, 282, 299, 302, 380, 1560, 1650, 2235, 2277, 2400, 2562, 2565, 2729, 2823, 3160, 3181, 3946, 4210, 4294, 4309, 4405, 4580, 5069, 5159, 5189, 5231, 5331, 5519, 5737, 5882, 5993, 5996, 6011, 6312, 6522, 6585, 6606, 7316, 7394, 7819, 7904, 7966, 7998, 8534, 8648, 8946, 9339, 9354, 9530, 9842, 9966, 10041, 10250, 10416, 10445, 10484, 10527, 10874, 11193, 11292, 11505, 11568, 11655, 11745, 11838, 12078, 12180, 12222, 12697, 12745, 12819, 12880, 12935, 13011, 13087, 13256, 13415, 13526, 13813, 14259, 15385, 15473, 16192, 17101, 17437, 17755, 17936, 17989, 18016, 18040, 18727, 18853, 18911, 19232, 19263, 19310, 19833, 19929, 19932, 19947, 19980, 20793, 20802, 21305, 21528, 21627, 21684, 22414, 22660, 22670, 22787, 23326, 24063, 24145, 24409, 24595, 24681, 25120, 25139, 25210, 25381, 25384, 25648, 25664, 25773, 25987, 26196, 26208, 26576, 26587, 26653, 26736, 27892, 27967, 28042, 28622, 28840, 28

In [10]:
def phred33ToQ(qual):
    return [ord(c) - 33 for c in qual]

In [11]:
sequences, quality = readFastq('ERR037900_1.first1000.fastq')
# Report which sequencing cycle has the problem.  Remember that a sequencing cycle corresponds to a particular offset in all the reads. For example, if the leftmost read position seems to have a problem consistently across reads, report 0. If the fourth position from the left has the problem, report 3.
# Initialize an array to store the sum of quality scores and total counts for each cycle
# (Assuming a max read length of 200, but it dynamically adjusts)
max_len = max(len(q) for q in quality)
quality_sums = [0] * max_len
read_counts = [0] * max_len

# Step 1: Accumulate quality scores for each cycle position
for q in quality:
    q_scores = phred33ToQ(q)
    for i in range(len(q_scores)):
        quality_sums[i] += q_scores[i]
        read_counts[i] += 1

# Step 2: Calculate the average quality for each cycle and find the problematic one
problematic_cycle = None
for i in range(max_len):
    if read_counts[i] > 0:
        avg_quality = quality_sums[i] / read_counts[i]
        
        # Check if the average quality for this specific cycle falls below your threshold
        if avg_quality < 10: 
            problematic_cycle = i
            print(f"Problematic sequencing cycle found at offset: {i} (Average Phred Score: {avg_quality:.2f})")
            break # Remove 'break' if you want to see if multiple cycles have systemic issues

if problematic_cycle is None:
    print("No systemic problematic sequencing cycle detected with an average Phred score < 10.")

Problematic sequencing cycle found at offset: 66 (Average Phred Score: 4.53)
